# Producers & Delivery: acks, Idempotence, and Retries Under Failure

Run this notebook cell-by-cell against the live cluster spawned from the topic page. See `concept.md` for the what/why background.

**This topic runs no Spark job.** Most cells talk to the brokers directly over the network (`kafka-1:9092` / `kafka-2:9092` / `kafka-3:9092`, reachable in-cluster) using `kafka-python`, but one section (`acks=all` + idempotence) instead prints a command for you to run in a **host terminal** -- the driver container this Jupyter kernel runs in has no Docker CLI/socket access, the same constraint `kafka-architecture-kraft` documents for its `docker exec` steps. `concept.md` explains why that section can't use `kafka-python`.

## 0. Create a single-partition demo topic (host terminal)

Every run below needs to go through the *same* leader broker, so a single `docker restart` reliably affects every message in flight rather than just whichever partition happened to get hit. This cluster's auto-create default (`KAFKA_NUM_PARTITIONS=3`, same as `kafka-topics-partitions`) would spread messages across 3 partitions/leaders, so this topic is created explicitly with **1 partition** instead. In your host terminal, run:

```
docker exec spark-kafka-1 /opt/kafka/bin/kafka-topics.sh --bootstrap-server localhost:9092 --create --topic producers-delivery-demo --partitions 1 --replication-factor 3
```

In [ ]:
print("docker exec spark-kafka-1 /opt/kafka/bin/kafka-topics.sh "
      "--bootstrap-server localhost:9092 --create --topic producers-delivery-demo "
      "--partitions 1 --replication-factor 3")

## 1. Find the topic's leader broker (host terminal)

The induced-failure steps below restart *the broker leading this topic's one partition* -- run the describe command and note the `Leader` broker ID (1, 2, or 3) before continuing:

```
docker exec spark-kafka-1 /opt/kafka/bin/kafka-topics.sh --bootstrap-server localhost:9092 --describe --topic producers-delivery-demo
```

You'll restart `spark-kafka-<that ID>` (not necessarily `spark-kafka-1`) in the next two sections.

In [ ]:
print("docker exec spark-kafka-1 /opt/kafka/bin/kafka-topics.sh "
      "--bootstrap-server localhost:9092 --describe --topic producers-delivery-demo")

## 2. `acks=0` -- fire-and-forget, loss possible and undetectable

The cell below sends 60 messages with `acks=0` (and `retries=0`, so nothing is retried) over about 30 seconds. **Partway through -- once you see it printing -- switch to your host terminal and run:**

```
docker restart spark-kafka-<leader ID from step 1>
```

Then come back and let the cell finish. The cell after it re-reads the topic and compares how many messages were *attempted* against how many actually *landed* -- any gap is loss `acks=0` never told the producer about.

In [ ]:
import time

from kafka import KafkaConsumer, KafkaProducer, TopicPartition

BOOTSTRAP_SERVERS = ["kafka-1:9092", "kafka-2:9092", "kafka-3:9092"]
TOPIC = "producers-delivery-demo"
ACKS0_COUNT = 60

producer = KafkaProducer(bootstrap_servers=BOOTSTRAP_SERVERS, acks=0, retries=0)
print(f"Sending {ACKS0_COUNT} acks=0 messages over ~30s -- restart the leader broker now.")

acks0_send_errors = 0
for seq in range(ACKS0_COUNT):
    try:
        producer.send(TOPIC, value=f"acks0-msg-{seq}".encode())
    except Exception as exc:  # acks=0 doesn't wait for a broker response, but the
        acks0_send_errors += 1  # local socket/metadata call itself can still raise
        print(f"  seq={seq} raised locally: {exc!r}")
    time.sleep(0.5)

producer.flush()
producer.close()
print(f"Attempted {ACKS0_COUNT} sends, {acks0_send_errors} raised a local error.")

In [ ]:
consumer = KafkaConsumer(
    bootstrap_servers=BOOTSTRAP_SERVERS,
    consumer_timeout_ms=5000,
    auto_offset_reset="earliest",
)
tp = TopicPartition(TOPIC, 0)
consumer.assign([tp])
consumer.seek_to_beginning(tp)

acks0_landed = {msg.value.decode() for msg in consumer if msg.value.decode().startswith("acks0-msg-")}
consumer.close()

lost = ACKS0_COUNT - len(acks0_landed)
print(f"Sent {ACKS0_COUNT} acks=0 messages, {len(acks0_landed)} actually landed on the broker.")
if lost > 0:
    print(f"{lost} message(s) were lost -- acks=0 raised no error, so the producer never knew.")
else:
    print("None lost this run (the restart window can miss every in-flight message by timing "
          "luck) -- re-run this section with a later restart if you want to see a gap.")

## 3. `acks=all` + idempotence -- automatic retries, zero duplicates (CLI, host terminal)

`kafka-python==2.0.2` has no `enable_idempotence` config (see `concept.md`), so this step runs the broker's own `kafka-console-producer.sh` instead, from a **host terminal**. The command below feeds it 60 numbered lines, one every ~0.5s, matching the previous section's pacing -- copy it from the cell below, swapping in the leader broker if it isn't `spark-kafka-1`.

**Partway through, restart the same leader broker again, from another host terminal:**

```
docker restart spark-kafka-<leader ID from step 1>
```

In [ ]:
print(
    "docker exec spark-kafka-1 bash -c "
    "'for i in $(seq 0 59); do echo idem-msg-$i; sleep 0.5; done "
    "| /opt/kafka/bin/kafka-console-producer.sh "
    "--broker-list localhost:9092 --topic producers-delivery-demo "
    "--producer-property acks=all --producer-property enable.idempotence=true'"
)

Once the host-terminal command above has finished (all 60 lines produced and the broker restart completed), run the cell below to verify the result.

In [ ]:
IDEM_COUNT = 60

consumer = KafkaConsumer(
    bootstrap_servers=BOOTSTRAP_SERVERS,
    consumer_timeout_ms=5000,
    auto_offset_reset="earliest",
)
tp = TopicPartition(TOPIC, 0)
consumer.assign([tp])
consumer.seek_to_beginning(tp)

idem_landed = [msg.value.decode() for msg in consumer if msg.value.decode().startswith("idem-msg-")]
consumer.close()

idem_unique = set(idem_landed)
print(f"Sent {IDEM_COUNT} idempotent messages, {len(idem_landed)} landed, {len(idem_unique)} unique values.")
if len(idem_landed) == len(idem_unique) == IDEM_COUNT:
    print("Every message landed exactly once: acks=all + idempotence retried through the "
          "broker restart with zero duplicates and zero loss.")
else:
    print("Landed/unique count differs from sent count -- re-check the host-terminal run "
          "finished before this cell executed, then re-run this cell.")

## 4. `acks=1` -- the middle ground, measured without a repeat restart

A leader dying in the exact window between its own local write and follower replication is too narrow a race to reliably reproduce on demand, so this run is **not** paired with another live broker restart -- it establishes the normal, healthy-cluster baseline for `acks=1` so the summary below has a real number to compare against the two runs above. See `concept.md`'s "What it is" section for the specific failure window `acks=1` leaves open that `acks=all` closes.

In [ ]:
ACKS1_COUNT = 60

producer = KafkaProducer(bootstrap_servers=BOOTSTRAP_SERVERS, acks=1, retries=0)
for seq in range(ACKS1_COUNT):
    producer.send(TOPIC, value=f"acks1-msg-{seq}".encode())
producer.flush()
producer.close()

consumer = KafkaConsumer(
    bootstrap_servers=BOOTSTRAP_SERVERS,
    consumer_timeout_ms=5000,
    auto_offset_reset="earliest",
)
tp = TopicPartition(TOPIC, 0)
consumer.assign([tp])
consumer.seek_to_beginning(tp)

acks1_landed = {msg.value.decode() for msg in consumer if msg.value.decode().startswith("acks1-msg-")}
consumer.close()

print(f"Sent {ACKS1_COUNT} acks=1 messages, {len(acks1_landed)} landed.")

## 5. Summary: at-most-once vs. at-least-once vs. (practically) exactly-once

Three different `acks`/idempotence configurations, three different observed guarantees -- the table below prints this run's actual measured counts, not received wisdom.

In [ ]:
print(f"{'config':<28}{'sent':>6}{'landed':>8}  guarantee observed")
print("-" * 78)
print(f"{'acks=0':<28}{ACKS0_COUNT:>6}{len(acks0_landed):>8}  "
      f"at-most-once (loss possible, undetected by the producer)")
print(f"{'acks=all + idempotence':<28}{IDEM_COUNT:>6}{len(idem_landed):>8}  "
      f"(effectively) exactly-once on the producer side")
print(f"{'acks=1':<28}{ACKS1_COUNT:>6}{len(acks1_landed):>8}  "
      f"at-least-once in general (this run: no leader failure induced)")